In [0]:
%python
# ============================================================
# Modern Data Platform - Ecommerce
# Notebook 01 : Project Initialization
# ============================================================

CATALOG = "de_learning_workspace"

print("=" * 60)
print("Setting Catalog...")
print("=" * 60)

spark.sql(f"USE CATALOG {CATALOG}")

# # ------------------------------------------------------------
# # Create Schemas
# # ------------------------------------------------------------

# schemas = [
#     "source",
#     "bronze",
#     "silver",
#     "gold"
# ]

# for schema in schemas:
#     spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

# print("Schemas created successfully.")

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("=" * 60)
print("Available Schemas")
print("=" * 60)


In [0]:
# %sql




# CREATE EXTERNAL LOCATION landing_location
# URL 'abfss://landing@ecommercedatalake001.dfs.core.windows.net/'
# WITH (CREDENTIAL cred_adlsgen2_ext_locations);


# CREATE EXTERNAL LOCATION bronze_location
# URL 'abfss://bronze@ecommercedatalake001.dfs.core.windows.net/'
# WITH (CREDENTIAL cred_adlsgen2_ext_locations);


# CREATE EXTERNAL LOCATION silver_location
# URL 'abfss://silver@ecommercedatalake001.dfs.core.windows.net/'
# WITH (CREDENTIAL cred_adlsgen2_ext_locations);


# CREATE EXTERNAL LOCATION gold_location
# URL 'abfss://gold@ecommercedatalake001.dfs.core.windows.net/'
# WITH (CREDENTIAL cred_adlsgen2_ext_locations);


# CREATE EXTERNAL LOCATION checkpoint_location
# URL 'abfss://checkpoints@ecommercedatalake001.dfs.core.windows.net/'
# WITH (CREDENTIAL cred_adlsgen2_ext_locations);


# CREATE EXTERNAL LOCATION archive_location
# URL 'abfss://archive@ecommercedatalake001.dfs.core.windows.net/'
# WITH (CREDENTIAL cred_adlsgen2_ext_locations);





In [0]:
jdbc_hostname = "e-commerce-server.database.windows.net"
jdbc_port = 1433
jdbc_database = "e-commerce-db"

jdbc_url = (
    f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};"
    f"database={jdbc_database}"
)

username = dbutils.secrets.get(
    scope="ecommerce-scope",
    key="sql-username"
)

password = dbutils.secrets.get(
    scope="ecommerce-scope",
    key="sql-password"
)

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, max

def load_to_bronze(source_table_name, destination_table_name):

    watermark = (
        spark.read.table("de_learning_workspace.audit.watermark")
        .filter(col("table_name") == source_table_name)
        .select("last_processed_updated_at")
        .first()["last_processed_updated_at"]
    )

    query = f"""
    (
        SELECT *
        FROM {source_table_name}
        WHERE updated_at > '{watermark}'
    ) AS bronze_src
    """

    print(f"Loading {source_table_name}")

    df = (
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", query)
        .option("user", username)
        .option("password", password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load()
    )

    bronze_df = (
        df
        .withColumn("_load_timestamp", current_timestamp())
        .withColumn("_source_system", lit("azure_sql"))
    )

    (
        bronze_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(destination_table_name)
    )

    if bronze_df.head(1):

        max_updated_at = (
            bronze_df
            .agg(max("updated_at").alias("max_updated_at"))
            .first()["max_updated_at"]
        )

        spark.sql(f"""
            UPDATE de_learning_workspace.audit.watermark
            SET last_processed_updated_at = TIMESTAMP('{max_updated_at}')
            WHERE table_name = '{source_table_name}'
        """)

    print(f"Completed loading {destination_table_name}")

In [0]:
tables = [
    ("dbo.customers", "de_learning_workspace.bronze.customers"),
    ("dbo.products", "de_learning_workspace.bronze.products"),
    ("dbo.orders", "de_learning_workspace.bronze.orders"),
    ("dbo.order_items", "de_learning_workspace.bronze.order_items"),
    ("dbo.inventory", "de_learning_workspace.bronze.inventory"),
    ("dbo.payments", "de_learning_workspace.bronze.payments"),
    ("dbo.shipments", "de_learning_workspace.bronze.shipments")
]

In [0]:

for source_table, destination_table in tables:
    load_to_bronze(
        source_table,
        destination_table
    )